In [ ]:
import numpy as np
from collections import Counter

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None

In [ ]:
class DecisionTreeClassifier:
    def __init__(self, max_depth=10, min_samples_split=2, criterion="gini", max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.criterion = criterion
        self.max_features = max_features
        self.root = None

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        self.n_features_ = X.shape[1]
        self.root = self._grow_tree(X, y, depth=0)

    def predict(self, X):
        X = np.array(X)
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _grow_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))

        if depth >= self.max_depth or n_classes == 1 or n_samples < self.min_samples_split:
            return Node(value=self._most_common_label(y))

        feat_idxs = self._get_feature_indices(n_features)
        best_feature, best_threshold = self._best_split(X, y, feat_idxs)

        if best_feature is None:
            return Node(value=self._most_common_label(y))

        left_idxs, right_idxs = self._split(X[:, best_feature], best_threshold)

        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)

        return Node(feature=best_feature, threshold=best_threshold, left=left, right=right)

    def _get_feature_indices(self, n_features):
        if self.max_features is None:
            return np.arange(n_features)

        if isinstance(self.max_features, str):
            if self.max_features == "sqrt":
                k = max(1, int(np.sqrt(n_features)))
            elif self.max_features == "log2":
                k = max(1, int(np.log2(n_features)))
            else:
                k = n_features
        else:
            k = min(n_features, int(self.max_features))

        return np.random.choice(n_features, k, replace=False)

    def _best_split(self, X, y, feature_indices):
        best_gain = -1
        split_idx = None
        split_threshold = None

        for feature_idx in feature_indices:
            X_column = X[:, feature_idx]
            thresholds = self._candidate_thresholds(X_column)

            for threshold in thresholds:
                gain = self._information_gain(y, X_column, threshold)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feature_idx
                    split_threshold = threshold

        return split_idx, split_threshold

    def _candidate_thresholds(self, X_column):
        values = np.unique(X_column)
        if len(values) <= 1:
            return values
        return (values[:-1] + values[1:]) / 2

    def _information_gain(self, y, X_column, threshold):
        parent_impurity = self._impurity(y)
        left_idxs, right_idxs = self._split(X_column, threshold)

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        n = len(y)
        n_left, n_right = len(left_idxs), len(right_idxs)

        left_impurity = self._impurity(y[left_idxs])
        right_impurity = self._impurity(y[right_idxs])

        child_impurity = (n_left / n) * left_impurity + (n_right / n) * right_impurity

        return parent_impurity - child_impurity

    def _impurity(self, y):
        if self.criterion == "entropy":
            return self._entropy(y)
        return self._gini(y)

    def _gini(self, y):
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / counts.sum()
        return 1 - np.sum(probabilities ** 2)

    def _entropy(self, y):
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / counts.sum()
        probabilities = probabilities[probabilities > 0]
        return -np.sum(probabilities * np.log2(probabilities))

    def _split(self, X_column, threshold):
        left_idxs = np.argwhere(X_column <= threshold).flatten()
        right_idxs = np.argwhere(X_column > threshold).flatten()
        return left_idxs, right_idxs

    def _most_common_label(self, y):
        return Counter(y).most_common(1)[0][0]

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [ ]:
class RandomForestClassifier:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, criterion="gini", max_features="sqrt"):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.criterion = criterion
        self.max_features = max_features
        self.trees = []

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        self.trees = []

        for _ in range(self.n_trees):
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                criterion=self.criterion,
                max_features=self.max_features
            )
            X_sample, y_sample = self._bootstrap_sample(X, y)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        X = np.array(X)
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        return np.array([self._most_common_label(preds) for preds in tree_preds])

    def score(self, X, y):
        y_pred = self.predict(X)
        return np.mean(y_pred == np.array(y))

    def _bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, n_samples, replace=True)
        return X[idxs], y[idxs]

    def _most_common_label(self, y):
        return Counter(y).most_common(1)[0][0]

In [ ]:
X = np.array([
    [2.7, 2.5],
    [1.3, 1.8],
    [3.0, 3.2],
    [1.2, 0.9],
    [3.1, 2.9],
    [1.0, 1.1],
    [2.9, 3.0],
    [1.4, 1.3]
])

y = np.array([1, 0, 1, 0, 1, 0, 1, 0])

In [ ]:
rf = RandomForestClassifier(
    n_trees=20,
    max_depth=5,
    min_samples_split=2,
    criterion="gini",
    max_features="sqrt"
)

rf.fit(X, y)
predictions = rf.predict(X)
accuracy = rf.score(X, y)

print("Predictions:", predictions)
print("Accuracy:", accuracy)

In [ ]:
X_new = np.array([
    [2.8, 2.7],
    [1.1, 1.0],
    [3.2, 3.1]
])

print(rf.predict(X_new))